# Graph-Theoretic Synchronization with Literature-Supported QUAD Parameters

This notebook implements the graph-theoretic coupling-strength allocation method on a **random weakly connected directed graph with exactly one source SCC**, so the generated graph has a directed spanning tree.

The oscillator menu now contains only systems for which the parameter used by the allocation is supported by a published analytical bound:

- **Lorenz**
- **Chua**
- **Rössler-like (piecewise-linear)**

The standard Rössler oscillator has been removed. The Physica A synchronization paper itself notes that x-coupled Rössler attractors need not satisfy the required global assumption, and I did not find a published numerical global QUAD tuple for the standard Rössler model compatible with the condition used here.

## Corrected imbalance condition

The implementation uses

$$
d_i\le -2a,\qquad i\ne r,
$$

so every path contribution in the construction of $W^-$ is scaled by **$2a$**, not by $a$.

## Literature used for the oscillator constants

1. **Lorenz:** A.B. Sahaya Arokiadoss and G. Arunkumar, *A (purely) graph-theoretic approach to synchronization of nonlinear dynamical networks*, Physica A 692 (2026) 131508. The simulation section gives the scalar $a$ used below.
2. **Chua:** X. Liu and T. Chen, *Synchronization of identical neural networks and other systems with an adaptive coupling strength*, International Journal of Circuit Theory and Applications 38 (2010) 631–648. For Chua's circuit they explicitly report
   $$
   f\in\mathrm{QUAD}(I_3,10I_3,0.6218).
   $$
   A later Physica A paper on successive lag synchronization uses the common Chua parameters $\alpha=15.6,\beta=28,m_0=-1.143,m_1=-0.714$ and states the same QUAD tuple.
3. **Rössler-like:** W. Lin and G. Chen, *Using white noise to enhance synchronization of coupled chaotic systems*, Chaos 16, 013134 (2006), and subsequent papers using the same piecewise-linear Rössler-like oscillator. A published global one-sided/Lipschitz bound is $L=0.4926$. Using the QUAD/Lipschitz theorem of DeLellis--di Bernardo--Russo, full-state coupling with $a=1$ gives the explicit bound
   $$
   (x-y)^\top[f(x)-f(y)-I_3(x-y)]
   \le -0.5074\|x-y\|^2.
   $$


In [ ]:
# Colab/Jupyter dependencies
!pip install -q numpy scipy pandas matplotlib networkx ipywidgets openpyxl

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import ipywidgets as widgets
from IPython.display import display, clear_output

%matplotlib inline
plt.close("all")
warnings.filterwarnings("ignore", category=RuntimeWarning)

## 1) Oscillators and explicit QUAD/system parameters

The synchronization condition required by the graph-theoretic algorithm is

$$
(x-y)^\top\!\left[f(x)-f(y)-aP(x-y)\right]
\le -c\|x-y\|_2^2,
$$

for a diagonal $0$--$1$ matrix $P$, some $a>0$, and some $c>0$.

### Lorenz

The Lorenz system is

$$
\dot x=\sigma(y-x),\qquad
\dot y=x(\rho-z)-y,\qquad
\dot z=xy-\beta z,
$$

with

$$
\sigma=10,\qquad \rho=28,\qquad \beta=\frac83.
$$

The 2026 Physica A paper uses x-coupling,

$$
P=\operatorname{diag}(1,0,0),
$$

and the explicit scalar

$$
a=
\frac{\beta(\beta+1)(\rho+\sigma)^2}{16(\beta-1)}
-\sigma
=519.466666\ldots.
$$

That is the value used by the notebook. The allocation algorithm only needs $a$.

### Chua

The implemented Chua circuit is

$$
\begin{aligned}
\dot x_1&=15.6\,[x_2-x_1-h(x_1)],\\
\dot x_2&=x_1-x_2+x_3,\\
\dot x_3&=-28x_2,
\end{aligned}
$$

where

$$
h(s)=m_1s+\frac12(m_0-m_1)(|s+1|-|s-1|),
$$

with

$$
m_0=-1.143,\qquad m_1=-0.714.
$$

For this standard parameter set, the published QUAD choice is

$$
\boxed{
P=I_3,\qquad a=10,\qquad c=0.6218.
}
$$

Thus

$$
(x-y)^\top[f(x)-f(y)-10(x-y)]
\le -0.6218\|x-y\|^2.
$$

This replaces the previous notebook's internally estimated Chua bound.

### Piecewise-linear Rössler-like oscillator

The standard Rössler oscillator is **not** used. Instead we use the globally Lipschitz piecewise-linear Rössler-like chaotic oscillator

$$
\begin{aligned}
\dot x_1&=-\alpha(\Gamma x_1+\beta x_2+\lambda x_3),\\
\dot x_2&=\alpha(x_1+\gamma x_2),\\
\dot x_3&=\alpha\mu[\psi(x_1)-x_3],
\end{aligned}
$$

with

$$
\psi(s)=
\begin{cases}
0, & s<2.56,\\
\xi(s-2.56), & s\ge 2.56,
\end{cases}
$$

and

$$
\alpha=0.03,\quad
\beta=1.5,\quad
\gamma=0.2,\quad
\mu=1.5,\quad
\lambda=0.75,\quad
\xi=21.43,\quad
\Gamma=0.075.
$$

For this oscillator the literature reports the global one-sided/Lipschitz bound

$$
(x-y)^\top[f(x)-f(y)]
\le 0.4926\,\|x-y\|^2.
$$

Therefore, with full-state coupling $P=I_3$, choosing $a=1$ gives

$$
(x-y)^\top[f(x)-f(y)-a(x-y)]
\le -(1-0.4926)\|x-y\|^2,
$$

so a valid explicit QUAD choice for the notebook is

$$
\boxed{
P=I_3,\qquad a=1,\qquad c=0.5074.
}
$$

Unlike the previous standard-Rössler option, this value is **not estimated from a simulated trajectory**.


In [ ]:
def lorenz_oscillator(t, U, sigma=10.0, rho=28.0, beta=8/3):
    dU = np.empty_like(U)
    x, y, z = U
    dU[0] = sigma * (y - x)
    dU[1] = x * (rho - z) - y
    dU[2] = x * y - beta * z
    return dU


def chua_nonlinearity(x, m0=-1.143, m1=-0.714):
    return m1*x + 0.5*(m0-m1)*(np.abs(x+1.0)-np.abs(x-1.0))


def chua_oscillator(t, U, alpha=15.6, beta=28.0,
                    m0=-1.143, m1=-0.714):
    dU = np.empty_like(U)
    x, y, z = U
    h = chua_nonlinearity(x, m0=m0, m1=m1)
    dU[0] = alpha*(y - x - h)
    dU[1] = x - y + z
    dU[2] = -beta*y
    return dU


def rossler_like_nonlinearity(x, xi=21.43, threshold=2.56):
    return np.where(x < threshold, 0.0, xi*(x-threshold))


def rossler_like_oscillator(t, U,
                            alpha=0.03,
                            beta=1.5,
                            gamma=0.2,
                            mu=1.5,
                            lam=0.75,
                            xi=21.43,
                            Gamma=0.075):
    """
    Piecewise-linear Rössler-like chaotic oscillator used in the
    synchronization literature. This is NOT the standard Rössler system.
    """
    dU = np.empty_like(U)
    x1, x2, x3 = U
    psi = rossler_like_nonlinearity(x1, xi=xi, threshold=2.56)

    dU[0] = -alpha*(Gamma*x1 + beta*x2 + lam*x3)
    dU[1] = alpha*(x1 + gamma*x2)
    dU[2] = alpha*mu*(psi - x3)
    return dU


def lorenz_theory_a(sigma=10.0, rho=28.0, beta=8/3):
    return (
        beta*(beta+1.0)*(rho+sigma)**2
        /(16.0*(beta-1.0))
        - sigma
    )


def oscillator_configuration(name, seed=0):
    key = name.lower()

    if key == 'lorenz':
        a = float(lorenz_theory_a())
        return dict(
            name='Lorenz',
            f=lorenz_oscillator,
            P=np.diag([1.0, 0.0, 0.0]),
            P_text='diag(1,0,0)',
            a=a,
            c_text='c > 0 (the 2026 simulation uses the explicit a shown here)',
            basis=(
                'Published analytic scalar a from the 2026 Physica A '
                'graph-theoretic synchronization paper'
            ),
            status='published theory-based parameter',
            base_ic=np.array([1.0, 1.0, 1.0]),
        )

    if key == 'chua':
        return dict(
            name='Chua',
            f=chua_oscillator,
            P=np.eye(3),
            P_text='I_3',
            a=10.0,
            c_text='0.6218',
            basis=(
                'Published explicit QUAD tuple '
                'QUAD(I_3, 10 I_3, 0.6218)'
            ),
            status='published global QUAD bound',
            base_ic=np.array([0.1, 0.2, 0.3]),
        )

    if key in ('rossler-like', 'rössler-like', 'rossler like', 'rössler like'):
        L_osl = 0.4926
        a = 1.0
        c = a - L_osl
        return dict(
            name='Rössler-like',
            f=rossler_like_oscillator,
            P=np.eye(3),
            P_text='I_3',
            a=a,
            c_text=f'{c:.4f}',
            basis=(
                'Published global one-sided/Lipschitz bound L=0.4926; '
                'with P=I_3 and a=1, DeLellis et al. gives c=1-L=0.5074'
            ),
            status='rigorous bound derived directly from published global constant',
            base_ic=np.array([-1.0, -1.0, -1.0]),
        )

    raise ValueError('Unknown oscillator')


def quad_parameter_table(seed=7):
    rows = []
    for name in ['Lorenz', 'Chua', 'Rössler-like']:
        cfg = oscillator_configuration(name, seed=seed)
        rows.append({
            'Oscillator': cfg['name'],
            'P': cfg['P_text'],
            'a used': cfg['a'],
            'Delta = aP': f"{cfg['a']:.6f} × {cfg['P_text']}",
            'c / omega': cfg['c_text'],
            'How obtained': cfg['basis'],
            'Guarantee': cfg['status'],
        })
    return pd.DataFrame(rows)


### Numerical parameters used

All entries below are now **literature-supported**. There is no sampled-attractor estimate in this version.


In [ ]:
quad_parameter_table(seed=7)

## 2) What is IC spread?

**IC spread** means **initial-condition spread**. It is not a graph parameter and it is not part of the theorem.

For oscillator $i$ the notebook uses

$$
z_i(0)=z_{\rm base}+\sigma_{\rm IC}\,\xi_i,
\qquad \xi_i\sim\mathcal N(0,I_3),
$$

where the UI value is $\sigma_{\rm IC}$.

- Smaller IC spread: oscillators start closer together.
- Larger IC spread: oscillators start farther apart.
- IC spread $=0$: all oscillators start identically, so synchronization would be trivial.

The default value is $0.05$.

## 3) Random weakly connected digraph generation

The generator first chooses a random number of SCCs and a random partition of the vertices. Each SCC is made strongly connected with a random directed Hamiltonian cycle plus extra random internal arcs. The SCCs are then connected by a random rooted acyclic condensation graph, with extra forward inter-SCC arcs.

This guarantees:

- weak connectivity;
- exactly one source SCC;
- a directed spanning tree;
- random, generally asymmetric connectivity.

Changing the **Random seed** changes both the graph and the initial conditions.

In [ ]:
def random_positive_composition(N,k,rng):
    if k==1:
        return [N]
    cuts=np.sort(rng.choice(np.arange(1,N),size=k-1,replace=False))
    return np.diff(np.r_[0,cuts,N]).tolist()


def add_random_strongly_connected_component(G,nodes,rng):
    nodes=list(nodes); m=len(nodes)
    if m<=1:
        return
    if m==2:
        G.add_edge(nodes[0],nodes[1]); G.add_edge(nodes[1],nodes[0]); return
    order=list(rng.permutation(nodes))
    for i in range(m):
        G.add_edge(order[i],order[(i+1)%m])
    p_extra=min(0.18,2.5/max(m-1,1))
    for u in nodes:
        for v in nodes:
            if u!=v and not G.has_edge(u,v) and rng.random()<p_extra:
                G.add_edge(u,v)


def generate_random_weakly_connected_digraph(N,seed=0):
    if N<3:
        raise ValueError('Use N >= 3')
    rng=np.random.default_rng(seed)
    G=nx.DiGraph(); G.add_nodes_from(range(N))
    # Use several SCCs, but keep the source SCC modest in size.  The
    # source-SCC cycle-space scaling in equation (8) is intentionally
    # conservative and can become extremely large for one huge source SCC.
    if N < 8:
        k = 2
    else:
        min_sccs = max(3, N // 18)
        max_sccs = min(10, max(min_sccs, N // 8))
        k = int(rng.integers(min_sccs, max_sccs + 1))

    max_source = min(8, N - (k - 1))
    min_source = 2 if max_source >= 2 else 1
    source_size = int(rng.integers(min_source, max_source + 1))

    if k == 1:
        sizes = [N]
    else:
        remaining = N - source_size
        tail_sizes = random_positive_composition(remaining, k - 1, rng)
        sizes = [source_size] + tail_sizes

    labels=list(rng.permutation(np.arange(N)))
    blocks=[]; p=0
    for s in sizes:
        blocks.append(labels[p:p+s]); p+=s
    for block in blocks:
        add_random_strongly_connected_component(G,block,rng)
    # Rooted condensation tree: SCC 0 is unique source.
    for child in range(1,k):
        parent=int(rng.integers(0,child))
        G.add_edge(int(rng.choice(blocks[parent])),int(rng.choice(blocks[child])))
    # Extra forward inter-SCC arcs preserve condensation acyclicity.
    for i in range(k):
        for j in range(i+1,k):
            if rng.random()<0.35:
                for _ in range(int(rng.integers(1,3))):
                    G.add_edge(int(rng.choice(blocks[i])),int(rng.choice(blocks[j])))
    assert nx.is_weakly_connected(G)
    sccs=list(nx.strongly_connected_components(G))
    C=nx.condensation(G,sccs)
    assert sum(C.in_degree(v)==0 for v in C)==1
    return G

## 4) Coupling-strength allocation for the random SCCs

For each SCC, choose a root $v_r$. In a non-source SCC the root is chosen among vertices receiving inter-SCC arcs, preferring the vertex with the largest number of such incoming arcs.

For each non-root vertex $v_j$, choose a directed shortest path

$$
v_r=v_{k_0}\to v_{k_1}\to\cdots\to v_{k_\ell}=v_j.
$$

The unscaled path vector receives the decreasing weights

$$
\ell,\ \ell-1,\ \ldots,\ 1.
$$

After summing the path vectors, the result is scaled by **$2a$**. Thus every non-root vertex receives imbalance at most $-2a$.

To make all internal arc weights positive without changing the imbalances, the notebook constructs a positive cycle-space vector. For every internal edge $(u,v)$ it finds a return path $v\leadsto u$ and adds that directed cycle. The sum is a positive circulation $X^0$ satisfying $QX^0=0$.

For the source SCC, $X^0$ is scaled with equation (8) of the paper. For every non-source SCC it is scaled by $1$.

Finally, Algorithm 2 assigns the inter-SCC incoming weights. If the positive-imbalance root has internal imbalance $d_1$, the total incoming inter-SCC weight at the root is $a+d_1/2$; for every other receiving vertex it is $a$.

In [ ]:
def ordered_scc_information(G):
    sccs=list(nx.strongly_connected_components(G))
    C=nx.condensation(G,sccs)
    sources=[v for v in C if C.in_degree(v)==0]
    if len(sources)!=1:
        raise ValueError('Graph must have exactly one source SCC')
    order=list(nx.topological_sort(C))
    return [set(C.nodes[c]['members']) for c in order], C


def choose_scc_root(G,nodes,is_source,rng):
    nodes=set(nodes)
    if is_source:
        return int(rng.choice(sorted(nodes)))
    count={v:0 for v in nodes}
    for u,v in G.edges():
        if v in nodes and u not in nodes:
            count[v]+=1
    cand=[v for v in nodes if count[v]>0]
    m=max(count[v] for v in cand)
    best=[v for v in cand if count[v]==m]
    return int(rng.choice(sorted(best)))


def construct_Wminus_for_scc(G,nodes,root,a):
    H=G.subgraph(nodes)
    W={(u,v):0.0 for u,v in H.edges()}
    lengths=[]
    for target in sorted(nodes):
        if target==root:
            continue
        path=nx.shortest_path(H,root,target)
        ell=len(path)-1; lengths.append(ell)
        for i,(u,v) in enumerate(zip(path[:-1],path[1:])):
            W[(u,v)] += float(ell-i)
    for e in W:
        W[e] *= 2.0*a
    return W,lengths


def positive_cycle_space_vector(G,nodes):
    H=G.subgraph(nodes)
    X={(u,v):0.0 for u,v in H.edges()}
    if len(nodes)<=1:
        return X
    cache={}
    for u,v in H.edges():
        if v not in cache:
            cache[v]=nx.single_source_shortest_path(H,v)
        ret=cache[v][u]
        X[(u,v)] += 1.0
        for e in zip(ret[:-1],ret[1:]):
            X[e] += 1.0
    return X


def internal_imbalance(nodes,weights):
    d={v:0.0 for v in nodes}
    for (u,v),w in weights.items():
        d[u]+=w; d[v]-=w
    return d


def allocate_intra_scc(G,nodes,root,a,is_source):
    nodes=set(nodes)
    if len(nodes)==1:
        return dict(root=root,path_lengths=[],path_sum=0,delta_w=0.0,
                    internal_imbalance={root:0.0})
    Wm,lengths=construct_Wminus_for_scc(G,nodes,root,a)
    X0=positive_cycle_space_vector(G,nodes)
    S=int(sum(lengths))
    delta=(2.0*a/len(nodes))*(1.0+S)*S if is_source else 1.0
    final={}
    for u,v in G.subgraph(nodes).edges():
        w=Wm[(u,v)] + delta*X0[(u,v)]
        G[u][v]['weight']=float(w); final[(u,v)]=float(w)
    return dict(root=root,path_lengths=lengths,path_sum=S,delta_w=float(delta),
                internal_imbalance=internal_imbalance(nodes,final))


def allocate_all_weights(G,a,seed=0):
    rng=np.random.default_rng(seed+100003)
    for u,v in G.edges():
        G[u][v]['weight']=0.0
    ordered,_=ordered_scc_information(G)
    diagnostics=[]
    for idx,nodes in enumerate(ordered):
        root=choose_scc_root(G,nodes,is_source=(idx==0),rng=rng)
        d=allocate_intra_scc(G,nodes,root,a,is_source=(idx==0))
        d.update(scc_index=idx,size=len(nodes),is_source=(idx==0),nodes=sorted(nodes))
        diagnostics.append(d)
    # Algorithm 2 for all non-source SCCs.
    for idx,nodes in enumerate(ordered[1:],start=1):
        nodes=set(nodes); root=diagnostics[idx]['root']
        d1=diagnostics[idx]['internal_imbalance'][root]
        groups={}
        for u,v in G.edges():
            if v in nodes and u not in nodes:
                groups.setdefault(v,[]).append((u,v))
        for v,edges in groups.items():
            total=a+0.5*d1 if v==root else a
            each=total/len(edges)
            for e in edges:
                G[e[0]][e[1]]['weight']=float(each)
        diagnostics[idx]['d1']=float(d1)
        diagnostics[idx]['incoming_inter_edges']=sum(map(len,groups.values()))
    return G,diagnostics


def verify_allocation(G,diagnostics,a,tol=1e-8):
    ordered,_=ordered_scc_information(G)
    rows=[]
    for idx,nodes in enumerate(ordered):
        root=diagnostics[idx]['root']; d=diagnostics[idx]['internal_imbalance']
        vals=[d[v] for v in nodes if v!=root]
        rows.append({
            'SCC':idx,'size':len(nodes),'root':root,
            'd_root (internal)':d[root],
            'max non-root d_i':max(vals) if vals else np.nan,
            'target':-2.0*a,
            'condition satisfied':all(x<=-2.0*a+tol for x in vals),
        })
    weights=np.array([data['weight'] for _,_,data in G.edges(data=True)])
    return dict(weakly_connected=nx.is_weakly_connected(G),
                number_of_sccs=len(ordered),
                all_arc_weights_positive=bool(np.all(weights>0)),
                min_arc_weight=float(weights.min()),
                max_arc_weight=float(weights.max()),
                imbalance_table=pd.DataFrame(rows))

## 5) Coupled simulation

The network is simulated in the standard diffusive form

$$
\dot z_i=f(z_i)+\sum_{j:(v_j,v_i)\in E}w_{ji}P(z_j-z_i).
$$

The weighted adjacency convention is $A_{ij}=w_{ji}$, so the indegree Laplacian is $L=D-A$.

In [ ]:
def graph_to_adjacency(G):
    N=G.number_of_nodes(); A=np.zeros((N,N))
    for u,v,data in G.edges(data=True):
        A[v,u]=data['weight']
    return A


def coupled_dynamics(t,u,f,A,indeg,P,N):
    U=u.reshape((3,N),order='F')
    dU=f(t,U)
    diff=U@A.T - U*indeg[np.newaxis,:]
    return (dU + P@diff).reshape(-1,order='F')


def simulate_network(G,cfg,N,t_end=10.0,seed=0,ic_spread=0.05,n_time=250):
    A=graph_to_adjacency(G); indeg=A.sum(axis=1)
    rng=np.random.default_rng(seed+200003)
    U0=cfg['base_ic'][:,None] + ic_spread*rng.normal(size=(3,N))
    y0=U0.reshape(-1,order='F')
    t=np.linspace(0.0,float(t_end),int(n_time))
    sol=solve_ivp(coupled_dynamics,(0.0,float(t_end)),y0,t_eval=t,
                  args=(cfg['f'],A,indeg,cfg['P'],N),method='LSODA',
                  rtol=1e-5,atol=1e-7)
    if not sol.success:
        raise RuntimeError(sol.message)
    return sol.y.T,sol.t,A,U0


def directed_spanning_tree_edges(G,root):
    T=nx.bfs_tree(G,root)
    if T.number_of_nodes()!=G.number_of_nodes():
        raise ValueError('Root does not reach all vertices')
    return list(T.edges())


def tree_edge_distances(X,N,edges):
    D=np.zeros((X.shape[0],len(edges)))
    for k in range(X.shape[0]):
        U=X[k].reshape((3,N),order='F')
        for q,(u,v) in enumerate(edges):
            D[k,q]=np.linalg.norm(U[:,u]-U[:,v])
    return D

## 6) Plotting functions

The plot format follows the supplied notebook: a circular directed-network plot followed by **“Total Pairwise Distance vs Time.”**

In [ ]:
def plot_network(G):
    N=G.number_of_nodes(); th=np.linspace(0,2*np.pi,N,endpoint=False)
    x=np.cos(th); y=np.sin(th)
    fig,ax=plt.subplots(figsize=(6,6))
    ax.scatter(x,y,s=60,c='lightblue',zorder=3)
    for u,v in G.edges():
        ax.annotate('',xy=(x[v],y[v]),xytext=(x[u],y[u]),
                    arrowprops=dict(arrowstyle='-|>',color='black',lw=0.8,
                                    alpha=0.65,shrinkA=6,shrinkB=6))
    ax.set_aspect('equal'); ax.axis('off'); plt.show()


def plot_pairwise_distances(t,D):
    fig,ax=plt.subplots(figsize=(10,6))
    for i in range(D.shape[1]):
        ax.plot(t,D[:,i],lw=1.5)
    ax.scatter(np.zeros(D.shape[1]),D[0,:],color='black',s=16,linewidths=0.5,zorder=3)
    ax.set_xlim(0,t.max()); ax.set_ylim(0,max(1e-12,1.08*D.max()))
    ax.set_xlabel('Time',fontsize=16); ax.set_ylabel('Pairwise Distances',fontsize=16)
    ax.set_title('Total Pairwise Distance vs Time',fontsize=18)
    ax.tick_params(labelsize=13); ax.grid(True); fig.tight_layout(); plt.show()

## 7) Interactive simulation UI

There is no topology UI. Every run automatically generates a random weakly connected digraph with one source SCC.

**IC spread $(\sigma)$** is the standard deviation of the Gaussian perturbation added to each oscillator's base initial condition.

In [ ]:
N_widget=widgets.IntSlider(
    value=100,min=5,max=200,step=5,description='Oscillators:',
    continuous_update=False,style={'description_width':'initial'}
)

osc_widget=widgets.Dropdown(
    options=['Lorenz','Chua','Rössler-like'],
    value='Lorenz',
    description='Oscillator:',
    style={'description_width':'initial'}
)

t_widget=widgets.FloatSlider(
    value=10.0,min=1.0,max=40.0,step=1.0,description='Final time:',
    continuous_update=False,style={'description_width':'initial'}
)

spread_widget=widgets.FloatLogSlider(
    value=0.05,base=10,min=-3,max=0,step=0.1,
    description='IC spread (σ):',
    continuous_update=False,
    style={'description_width':'initial'}
)

seed_widget=widgets.IntText(
    value=7,description='Random seed:',
    style={'description_width':'initial'}
)

run_button=widgets.Button(
    description='Run simulation',
    button_style='success',
    icon='play'
)

output=widgets.Output()

display(
    widgets.VBox([
        widgets.HBox([N_widget,osc_widget]),
        widgets.HBox([t_widget,spread_widget]),
        seed_widget,
        run_button,
        output
    ])
)


In [ ]:
def run_from_ui(_):
    with output:
        clear_output(wait=True)
        N=int(N_widget.value); osc=osc_widget.value; t_end=float(t_widget.value)
        seed=int(seed_widget.value); spread=float(spread_widget.value)

        print('🚀 Graph-Theoretic Synchronization Simulation')
        print('============================================')
        print(f'Oscillator: {osc}')
        print(f'N = {N}')
        print(f'Random seed = {seed}')
        print(f'IC spread = {spread:g} (Gaussian standard deviation around the base IC)')

        print('\nQUAD / system-parameter summary for all oscillator choices:')
        display(quad_parameter_table(seed=seed))

        cfg=oscillator_configuration(osc,seed=seed); a=cfg['a']
        print(f"\nSelected: {cfg['name']}")
        print(f"P = {cfg['P_text']}")
        print(f"a = {a:.6f}")
        print(f"c / omega = {cfg['c_text']}")
        print(f"How obtained: {cfg['basis']}")
        print(f"Status: {cfg['status']}")

        print('\n🎲 Generating random weakly connected digraph...')
        G=generate_random_weakly_connected_digraph(N,seed=seed)
        sccs,_=ordered_scc_information(G)
        print(f'Number of arcs = {G.number_of_edges()}')
        print(f'Number of SCCs = {len(sccs)}')
        print(f'SCC sizes = {[len(s) for s in sccs]}')
        print(f'Weakly connected = {nx.is_weakly_connected(G)}')

        print('\n🔧 Allocating coupling strengths...')
        G,diag=allocate_all_weights(G,a=a,seed=seed)
        checks=verify_allocation(G,diag,a=a)
        print(f"All arc weights positive = {checks['all_arc_weights_positive']}")
        print(f"Minimum arc weight = {checks['min_arc_weight']:.6g}")
        print(f"Maximum arc weight = {checks['max_arc_weight']:.6g}")
        print('\nCorrected imbalance check d_i <= -2a inside every SCC:')
        display(checks['imbalance_table'])

        print('\nFigure 1 — Random Connectivity Digraph')
        plot_network(G)

        print('⏱️ Simulating...')
        X,t,A,U0=simulate_network(G,cfg,N,t_end=t_end,seed=seed,ic_spread=spread)
        source_root=diag[0]['root']
        edges=directed_spanning_tree_edges(G,source_root)
        D=tree_edge_distances(X,N,edges)
        print('Figure 2 — Pairwise Distances Along a Directed Spanning Tree')
        plot_pairwise_distances(t,D)

        print(f'📊 Average initial tree-edge distance = {D[0].mean():.6e}')
        print(f'📊 Average final tree-edge distance   = {D[-1].mean():.6e}')
        print(f'📊 Maximum final tree-edge distance   = {D[-1].max():.6e}')

        globals()['LAST_RESULTS']=dict(G=G,diagnostics=diag,checks=checks,config=cfg,
                                       X=X,t=t,A=A,U0=U0,tree_edges=edges,pairwise_distances=D)
        print('✅ COMPLETE')

run_button.on_click(run_from_ui)
print("UI callback connected. Choose settings above and click 'Run simulation'.")

## 8) Optional export

Run this cell after a simulation to export the weighted adjacency matrix, arc list, spanning-tree distances, and SCC imbalance checks.

In [ ]:
if 'LAST_RESULTS' not in globals():
    print('Run a simulation from the UI first.')
else:
    R=LAST_RESULTS
    output_dir='/content/output' if os.path.isdir('/content') else './output'
    os.makedirs(output_dir,exist_ok=True)
    pd.DataFrame(R['A']).to_csv(os.path.join(output_dir,'adjacency_matrix.csv'),index=False)
    pd.DataFrame([{'tail':u,'head':v,'weight':d['weight']} for u,v,d in R['G'].edges(data=True)]).to_csv(
        os.path.join(output_dir,'weighted_arc_list.csv'),index=False)
    df=pd.DataFrame({'Time':R['t']})
    for q,(u,v) in enumerate(R['tree_edges']):
        df[f'd_{u}_{v}']=R['pairwise_distances'][:,q]
    df.to_excel(os.path.join(output_dir,'pairwise_distances.xlsx'),sheet_name='PairwiseDistances',index=False)
    R['checks']['imbalance_table'].to_csv(os.path.join(output_dir,'scc_imbalance_checks.csv'),index=False)
    print(f'Saved files to {output_dir}')

## Interpretation notes

1. **Why the graph is not an unrestricted weakly connected digraph:** weak connectivity alone does not imply a directed spanning tree. The random generator therefore enforces exactly one source SCC.
2. **Why only $N-1$ distances are plotted:** the paper checks differences along a directed spanning tree. If all those differences vanish, every node converges to the same trajectory.
3. **The corrected $-2a$ condition is enforced:** $W^-$ is scaled by $2a$, and the notebook displays a per-SCC check of $d_i\le-2a$.
4. **IC spread only controls the initial separation:** it does not alter the theorem or the parameter $a$ used in the coupling allocation.
5. **Standard Rössler has been removed:** no empirical QUAD estimate is used anywhere in this notebook.
6. **Chua uses a directly published QUAD tuple:** $P=I_3$, $a=10$, $c=0.6218$.
7. **Rössler-like uses a published global bound:** the literature reports $L=0.4926$ for the piecewise-linear Rössler-like system; choosing $a=1$ gives the explicit QUAD margin $c=0.5074$.
8. **Lorenz uses the scalar $a$ from the 2026 graph-theoretic synchronization paper:** this is exactly the quantity required by the coupling-allocation algorithm.
